# Samples and the ways they mislead

Core stage, lesson C5. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** tell a population from a sample and a parameter from a statistic, see how a random sample and a convenience sample differ as a number, and name selection, survivorship and nonresponse bias.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)

# One number for each order: the sum of its lines, quantity × unit price − discount.
# All 1,800 orders count: the canceled order and order 2075 (the line with the extra zero) are in.
price = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(amount=order_items["quantity"] * order_items["product_id"].map(price) - order_items["discount_usd"])
orders["value"] = orders["order_id"].map(lines.groupby("order_id")["amount"].sum())
ordered = pd.to_datetime(orders["ordered_at"])
orders["hour"] = ordered.dt.hour
orders["weekday"] = ordered.dt.day_name()
assert orders["value"].notna().all() and round(orders["value"].sum(), 2) == 22754.25
print(f"Data loaded: {len(orders):,} orders, {orders['value'].mean():.2f} dollars on average, {orders['value'].sum():,.2f} dollars in all.")

## Unit 1: Population, sample, parameter, statistic

**Predict first.** The sample is 200 of the 1,800 orders. Will its mean equal the mean of all 1,800 exactly, land close to it, or be far from it?

In [ ]:
value = orders["value"]                       # one number for each order
mu = value.mean()                             # the parameter: computed from all 1,800 orders
sample = value.sample(200, random_state=0)    # a simple random sample: 200 orders, none twice
xbar = sample.mean()                          # the statistic: computed from the sample
print(len(value), round(float(mu), 2), len(sample), round(float(xbar), 2))

**Predict first.** Draw the sample five times with seeds 0 to 4. Will the five sample means agree with each other?

In [ ]:
print([round(float(value.sample(200, random_state=seed).mean()), 2) for seed in range(5)])

## Unit 2: A random sample and a convenience sample

**Predict first.** Draw 1,000 samples of 200 orders in two ways: at random from all orders, and only from the 329 orders placed before 9 a.m. What will the average of the 1,000 sample means be for each way, to the nearest 0.1?

In [ ]:
mornings = orders[orders["hour"] < 9]["value"]    # the orders easiest to reach: placed before 9 a.m.
print(len(mornings), round(float(mornings.mean()), 2))
random_means = [value.sample(200, random_state=seed).mean() for seed in range(1000)]
morning_means = [mornings.sample(200, random_state=seed).mean() for seed in range(1000)]
print(round(float(np.mean(random_means)), 2), round(float(np.mean(morning_means)), 2))

**Predict first.** Subtract the parameter from each average. Then look at the smallest and the largest of each list of 1,000 means: do the two lists overlap, and does any morning mean reach the true mean of 12.64?

In [ ]:
print(round(float(np.mean(random_means) - mu), 2), round(float(np.mean(morning_means) - mu), 2))
print(round(float(np.min(random_means)), 2), round(float(np.max(random_means)), 2))
print(round(float(np.min(morning_means)), 2), round(float(np.max(morning_means)), 2))

## Unit 3: The people who are missing

**Predict first.** How many customers have a row in `visits`, and will their average number of visits be higher or lower than the average over all 150 customers?

In [ ]:
visits = orders.groupby("customer_id").size()                 # only customers who ordered have a row here
print(len(visits), int(visits.sum()), round(float(visits.mean()), 2))
all_visits = customers["customer_id"].map(visits).fillna(0)   # every customer, 0 visits for one who never ordered
print(len(all_visits), round(float(all_visits.mean()), 2))

**Predict first.** Suppose only the customers with 18 or more visits answer. How many answer, and how far is the average they report from the true average of all 150?

In [ ]:
# A made-up scenario, not in the data: the cafe asks all 150 customers "How often do you visit?",
# and suppose that only customers with 18 or more visits answer.
answered = all_visits[all_visits >= 18]
print(len(answered), round(float(answered.mean()), 2), round(float(all_visits.mean()), 2))

## Finish it

### Exercise 1: Finish it

**Task:** Who is missing from the orders table? The five customers who never ordered have no row in it. What share of customers is on the bronze tier among the customers who appear in the orders table, and what share among all 150 customers? Give both as a list, the first share first, each rounded to 3 decimals.

**Expected:** `answer_1` is a list of two numbers.

In [ ]:
tiers = customers.set_index("customer_id")["loyalty_tier"]   # given: the tier of each of the 150 customers
seen = tiers[tiers.index.isin(orders["customer_id"])]        # given: the tiers of the customers who appear in the orders table
# the share of bronze among `seen`, then among all customers, each rounded to 3 decimals
answer_1 = None  # replace None with [share among seen, share among all]

In [ ]:
_ = check("Exercise 1", "answer_1", [0.559, 0.567], traps={(0.567, 0.567): 'Both shares come from all 150 customers. The first share must use only the customers who appear in the orders table.'})

<details>
<summary>Hint</summary>

A comparison such as `seen == "bronze"` gives `True` and `False`, and the mean of those is the share of `True`.

</details>

<details>
<summary>Solution</summary>

```python
tiers = customers.set_index("customer_id")["loyalty_tier"]
seen = tiers[tiers.index.isin(orders["customer_id"])]
answer_1 = [round(float((seen == "bronze").mean()), 3), round(float((tiers == "bronze").mean()), 3)]
```

0.559 and 0.567: bronze is 81 of the 145 customers who appear and 85 of all 150, because four of the five customers who never ordered are bronze. The check cell reports 0.567 twice when both shares use all 150 customers.

</details>

## On your own

### Exercise 2: On your own

**Task:** Which weekday would give the most biased convenience sample of order values? Before you start, say in one sentence what the result looks like: a day name and one number. Group the orders by `weekday`, take each day's mean order value minus `mu`, and find the weekday whose gap is largest in size, whichever direction. Put the name and its gap, rounded to 2 decimals, in `answer_2` as a list: the name first. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of a weekday name and a number.

In [ ]:
_ = check("Exercise 2", "answer_2", ['Friday', 0.92], traps={('Thursday', -0.89): 'Thursday has the most negative gap. Which weekday has the largest gap in size, whichever direction?'})

<details>
<summary>Hint</summary>

`groupby("weekday")["value"].mean()` gives one mean for each day. Subtract `mu`, then ask for the largest absolute value with `.abs().idxmax()`.

</details>

<details>
<summary>Solution</summary>

```python
gap = orders.groupby("weekday")["value"].mean() - mu
day = gap.abs().idxmax()
answer_2 = [day, round(float(gap[day]), 2)]
```

Friday, 0.92. Thursday has a gap of -0.89, the largest in the other direction. One order explains most of Friday's gap: order 2075, the 197.00 order with the extra zero, was placed on a Friday, and without it the Friday mean is 12.76, 0.11 above `mu`.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wants the mean order value of all three cafes. They draw 100 orders at random from each store and average the three store means. Because the draws are random inside each store, that estimate centers on the plain average of the three store means, which the lines below compute. Fix the lines so that the estimate centers on the population mean, then set `answer_3` to it, rounded to 2 decimals.

**Expected:** `answer_3` is `12.64`.

In [ ]:
store_means = orders.groupby("store_id")["value"].mean()   # the mean order value of each store
answer_3 = round(float(store_means.mean()), 2)               # the colleague's estimate: each store counts the same

In [ ]:
_ = check("Exercise 3", "answer_3", 12.64, traps={12.7: 'Every store counts the same, but the stores hold 729, 716 and 355 orders. Should an order at Harbor weigh the same as an order at Old Town?'})

<details>
<summary>Hint</summary>

Print `orders.groupby("store_id").size()`. How many orders does each store hold, and how many of each are in the colleague's sample? Then look for `np.average` and its `weights`.

</details>

<details>
<summary>Solution</summary>

```python
store_means = orders.groupby("store_id")["value"].mean()
store_sizes = orders.groupby("store_id").size()
answer_3 = round(float(np.average(store_means, weights=store_sizes)), 2)
```

The colleague's estimate centers on 12.70, not 12.64. An order at Harbor had about twice the chance of being drawn (100 of 355) as an order at Old Town (100 of 729), so Harbor's higher mean counted too much. Weighting each store's mean by the orders it holds gives every order the same weight again. The check cell reports 12.7.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")